# Module 9 · 影片 3：案例 — 動作辨識推論（VideoMAE）

> **本節定位｜整合案例（全新）**
> 把影片 `01–02`（解碼、抽樣、前處理）整合，用預訓練 **VideoMAE** 對一段 clip
> 做**動作辨識**推論。CPU 可跑（單段真實短片）。

## 學習目標
- 完整跑一次影片推論流程：抽樣 16 影格 → processor → 模型 → 預測動作類別。
- 理解影片模型輸出 `logits (N, num_classes)` 的意義。
- 認識「凍結特徵」與「端到端微調」的差別（微調見 M11）。

<!-- concept-image:03_video_case_fig1 -->
<div align="center">
<img src="concept_images/03_video_case_fig1_pipeline_20260611_222401.png" alt="影片動作辨識端到端流程" width="640" style="max-width:100%;">
<br><sub>圖 1 · 影片動作辨識端到端流程</sub>
</div>

<!-- concept-image:03_video_case_fig2 -->
<div align="center">
<img src="concept_images/03_video_case_fig2_videomae_arch_20260611_222449.png" alt="VideoMAE 架構與時空注意力" width="640" style="max-width:100%;">
<br><sub>圖 2 · VideoMAE 架構與時空注意力</sub>
</div>

In [ ]:
import numpy as np
import av
from huggingface_hub import hf_hub_download

# 真實影片：均勻抽樣 16 張影格
def get_clip(num_frames=16):
    path = hf_hub_download(repo_id="nielsr/video-demo",
                           filename="eating_spaghetti.mp4", repo_type="dataset")
    container = av.open(path)
    frames = [f.to_ndarray(format="rgb24") for f in container.decode(video=0)]
    container.close()
    idx = np.linspace(0, len(frames) - 1, num_frames).astype(int)
    return [frames[i] for i in idx], "真實影片 eating_spaghetti.mp4"

clip_frames, source = get_clip()
print(f"資料來源：{source}")
print(f"clip 影格數 = {len(clip_frames)}，每張 {clip_frames[0].shape}")

## 1. 載入 VideoMAE 並推論

`videomae-base-finetuned-kinetics` 在 Kinetics-400（400 種動作）上微調過，可直接辨識動作。
需 `multimodal` extra；首次執行會下載權重。

<!-- concept-image:03_video_case_fig3 -->
<div align="center">
<img src="concept_images/03_video_case_fig3_transfer_learning_20260611_222536.png" alt="預訓練與微調的遷移學習" width="640" style="max-width:100%;">
<br><sub>圖 3 · 預訓練與微調的遷移學習</sub>
</div>

In [ ]:
try:
    import torch
    from transformers import AutoImageProcessor, VideoMAEForVideoClassification

    proc = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
    model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
    model.eval()

    inputs = proc(clip_frames, return_tensors="pt")
    print(f"模型輸入 pixel_values: {tuple(inputs['pixel_values'].shape)}  (N, T, C, H, W)")
    with torch.no_grad():
        logits = model(**inputs).logits                 # (N, num_classes=400)
    pred = logits.argmax(-1).item()
    print(f"輸出 logits 形狀: {tuple(logits.shape)}  (N, num_classes)")
    print(f"預測動作類別: {model.config.id2label[pred]}")
    print("（真實影片 → VideoMAE 預測出與『吃義大利麵』相關的動作。）")
except Exception as e:
    print("（未能下載 VideoMAE 權重，略過實際推論）", e)
    print("流程：抽樣 16 影格 → processor → VideoMAE → logits (N, 400) → argmax → 動作標籤。")

## 小結與延伸

| 重點 | 內容 |
|:--|:--|
| 流程 | 解碼 → 抽樣 16 影格 → processor → VideoMAE → `logits (N, num_classes)` |
| 預訓練資料 | Kinetics-400（400 種人類動作） |
| 輸入 | `(N, T, C, H, W)` 時空張量 |

**想用在自己的動作類別？** 把 VideoMAE **端到端微調**到你的資料集，
是 **Module 11 · `05_video_downstream`** 的內容。